# FastAPI 入门实战

## 为什么学 FastAPI?

深度学习模型训练好之后, 需要部署成服务供其他系统调用。
FastAPI 是目前最流行的 Python Web 框架之一:

- **快**: 性能接近 Go/Node.js
- **简单**: 代码量少, 上手快
- **自动文档**: 自动生成 Swagger UI
- **类型提示**: 基于 Python 类型注解, 自动校验
- **异步支持**: 原生 async/await

## 本课内容
1. FastAPI 基础
2. 路径参数与查询参数
3. 请求体与数据校验
4. 深度学习模型部署
5. 文件上传 (IQ数据)
6. 中间件与CORS
7. 完整RFFI服务示例

In [ ]:
# 安装 (如果还没安装)
# !pip install fastapi uvicorn pydantic

import fastapi
import pydantic
import uvicorn
print(f"FastAPI: {fastapi.__version__}")
print(f"Pydantic: {pydantic.__version__}")
print(f"Uvicorn: {uvicorn.__version__}")

## 1. FastAPI 基础

### 最简单的 FastAPI 应用

```python
# app.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello World"}
```

运行: `uvicorn app:app --reload`

访问:
- http://127.0.0.1:8000/ → API响应
- http://127.0.0.1:8000/docs → 自动生成的Swagger文档
- http://127.0.0.1:8000/redoc → ReDoc文档

In [ ]:
# 在Notebook中运行FastAPI (测试用)
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="My First API", version="1.0")

@app.get("/")
def root():
    return {"message": "Hello World"}

@app.get("/health")
def health_check():
    return {"status": "healthy"}

# 用TestClient测试 (不需要启动服务器)
client = TestClient(app)

response = client.get("/")
print(f"GET /: {response.json()}")

response = client.get("/health")
print(f"GET /health: {response.json()}")

In [ ]:
# HTTP方法
print("=== HTTP方法 ===")
print()
print("GET:    获取资源 (查询)")
print("POST:   创建资源 (提交数据)")
print("PUT:    更新资源 (全量替换)")
print("PATCH:  部分更新")
print("DELETE: 删除资源")
print()
print("RESTful API 设计:")
print("  GET    /devices       -> 获取所有设备")
print("  GET    /devices/1     -> 获取设备1")
print("  POST   /devices       -> 注册新设备")
print("  PUT    /devices/1     -> 更新设备1")
print("  DELETE /devices/1     -> 删除设备1")
print("  POST   /auth/verify   -> 认证设备")

---
## 2. 路径参数与查询参数

In [ ]:
app2 = FastAPI()

# 路径参数
@app2.get("/devices/{device_id}")
def get_device(device_id: int):
    return {"device_id": device_id, "name": f"Device-{device_id}"}

# 查询参数
@app2.get("/devices")
def list_devices(skip: int = 0, limit: int = 10, status: str = "active"):
    return {"skip": skip, "limit": limit, "status": status}

# 路径参数 + 查询参数
@app2.get("/devices/{device_id}/signals")
def get_device_signals(device_id: int, snr: float = 15.0, count: int = 10):
    return {"device_id": device_id, "snr": snr, "count": count}

client = TestClient(app2)

print("路径参数:")
print(f"  GET /devices/5: {client.get('/devices/5').json()}")
print()
print("查询参数:")
print(f"  GET /devices: {client.get('/devices').json()}")
print(f"  GET /devices?limit=5&status=offline: {client.get('/devices?limit=5&status=offline').json()}")
print()
print("路径+查询:")
print(f"  GET /devices/3/signals?snr=20: {client.get('/devices/3/signals?snr=20').json()}")

---
## 3. 请求体与数据校验

FastAPI 使用 Pydantic 模型自动校验请求数据。

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime

# 定义数据模型
class DeviceRegister(BaseModel):
    name: str = Field(..., min_length=1, max_length=50, description="设备名称")
    device_type: str = Field(..., pattern=r"^(wifi|ble|lora|zigbee)$", description="设备类型")
    location: Optional[str] = Field(None, description="设备位置")
    snr_threshold: float = Field(10.0, ge=0, le=50, description="SNR阈值")

class DeviceResponse(BaseModel):
    id: int
    name: str
    device_type: str
    location: Optional[str]
    registered_at: str

class AuthRequest(BaseModel):
    iq_data: List[float] = Field(..., min_length=8, description="IQ信号数据")
    sample_rate: float = Field(1e6, gt=0, description="采样率")
    expected_device: Optional[int] = Field(None, description="预期设备ID")

class AuthResponse(BaseModel):
    device_id: int
    confidence: float = Field(..., ge=0, le=1)
    is_authenticated: bool
    message: str

# 测试数据校验
device = DeviceRegister(name="Sensor-01", device_type="wifi", snr_threshold=15.0)
print(f"有效数据: {device.model_dump()}")

try:
    bad_device = DeviceRegister(name="", device_type="unknown")
except Exception as e:
    print(f"校验失败: {e}")

In [ ]:
app3 = FastAPI()
devices_db = {}
next_id = 1

@app3.post("/devices", response_model=DeviceResponse)
def register_device(device: DeviceRegister):
    global next_id
    device_record = DeviceResponse(
        id=next_id,
        name=device.name,
        device_type=device.device_type,
        location=device.location,
        registered_at=datetime.now().isoformat()
    )
    devices_db[next_id] = device_record
    next_id += 1
    return device_record

@app3.get("/devices", response_model=List[DeviceResponse])
def list_devices():
    return list(devices_db.values())

@app3.get("/devices/{device_id}", response_model=DeviceResponse)
def get_device(device_id: int):
    if device_id not in devices_db:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Device not found")
    return devices_db[device_id]

client = TestClient(app3)

# 注册设备
r1 = client.post("/devices", json={"name": "Sensor-A", "device_type": "wifi", "location": "Lab-1"})
print(f"注册: {r1.json()}")

r2 = client.post("/devices", json={"name": "Sensor-B", "device_type": "ble", "snr_threshold": 20.0})
print(f"注册: {r2.json()}")

# 列表
print(f"设备列表: {client.get('/devices').json()}")

# 查询
print(f"设备1: {client.get('/devices/1').json()}")

# 404
r404 = client.get("/devices/999")
print(f"设备999: status={r404.status_code}, {r404.json()}")

---
## 4. 深度学习模型部署

将训练好的模型部署为API服务。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 定义并训练一个简单的RFFI模型
class SimpleRFFNet(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# 训练模型
np.random.seed(42)
torch.manual_seed(42)
n_devices = 5
iq_len = 128

def gen_iq(dev_id, n=128, snr=15):
    np.random.seed(dev_id * 100)
    I = np.random.randn(n).astype(np.float32)
    Q = np.random.randn(n).astype(np.float32)
    I += (dev_id - 2) * 0.3
    Q += (dev_id - 2) * 0.2
    noise = np.random.randn(n).astype(np.float32) / (10**(snr/20))
    return np.stack([I + noise, Q + noise])

X, y = [], []
for d in range(n_devices):
    for _ in range(200):
        X.append(gen_iq(d, iq_len))
        y.append(d)
X = np.array(X)
y = np.array(y)
X = X / (np.std(X) + 1e-8)

model = SimpleRFFNet(n_classes=n_devices)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)
for epoch in range(50):
    logits = model(X_t)
    loss = criterion(logits, y_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    acc = (model(X_t).argmax(1) == y_t).float().mean().item()
print(f"模型训练完成, 准确率: {acc:.4f}")

# 保存模型
torch.save(model.state_dict(), "rffi_model.pth")
print("模型已保存: rffi_model.pth")

In [ ]:
# 部署模型为API
app_model = FastAPI(title="RFFI API", version="1.0")

# 启动时加载模型
rffi_model = SimpleRFFNet(n_classes=n_devices)
rffi_model.load_state_dict(torch.load("rffi_model.pth", weights_only=True))
rffi_model.eval()

class PredictRequest(BaseModel):
    iq_data: List[float] = Field(..., min_length=256, max_length=256, description="IQ数据(I,Q交替)")

class PredictResponse(BaseModel):
    device_id: int
    confidence: float
    probabilities: List[float]

@app_model.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    iq = np.array(req.iq_data, dtype=np.float32)
    iq = iq.reshape(2, iq_len)
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    
    with torch.no_grad():
        logits = rffi_model(x)
        probs = torch.softmax(logits, dim=1)
        device_id = probs.argmax(1).item()
        confidence = probs.max().item()
    
    return PredictResponse(
        device_id=device_id,
        confidence=round(confidence, 4),
        probabilities=[round(p, 4) for p in probs[0].numpy()]
    )

@app_model.get("/model/info")
def model_info():
    return {
        "model": "SimpleRFFNet",
        "n_classes": n_devices,
        "iq_length": iq_len,
        "input_format": "IQ interleaved [I0,Q0,I1,Q1,...]"
    }

# 测试
client = TestClient(app_model)

test_iq = gen_iq(2, iq_len)
iq_interleaved = []
for i in range(iq_len):
    iq_interleaved.extend([float(test_iq[0, i]), float(test_iq[1, i])])

r = client.post("/predict", json={"iq_data": iq_interleaved})
print(f"预测结果: {r.json()}")
print(f"模型信息: {client.get('/model/info').json()}")

In [ ]:
# 认证API (带阈值判断)
app_auth = FastAPI(title="RFFI Auth API")

rffi_model2 = SimpleRFFNet(n_classes=n_devices)
rffi_model2.load_state_dict(torch.load("rffi_model.pth", weights_only=True))
rffi_model2.eval()

class AuthRequest2(BaseModel):
    iq_data: List[float] = Field(..., min_length=256, max_length=256)
    claimed_device_id: int = Field(..., ge=0, lt=n_devices)
    confidence_threshold: float = Field(0.7, ge=0, le=1)

class AuthResponse2(BaseModel):
    authenticated: bool
    claimed_id: int
    predicted_id: int
    confidence: float
    reason: str

@app_auth.post("/auth/verify", response_model=AuthResponse2)
def verify_device(req: AuthRequest2):
    iq = np.array(req.iq_data, dtype=np.float32).reshape(2, iq_len)
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    
    with torch.no_grad():
        logits = rffi_model2(x)
        probs = torch.softmax(logits, dim=1)
        predicted_id = probs.argmax(1).item()
        confidence = probs.max().item()
    
    if confidence < req.confidence_threshold:
        return AuthResponse2(authenticated=False, claimed_id=req.claimed_device_id,
                            predicted_id=predicted_id, confidence=round(confidence, 4),
                            reason="Confidence below threshold (possible unknown device)")
    elif predicted_id != req.claimed_device_id:
        return AuthResponse2(authenticated=False, claimed_id=req.claimed_device_id,
                            predicted_id=predicted_id, confidence=round(confidence, 4),
                            reason="Predicted ID does not match claimed ID (possible spoofing)")
    else:
        return AuthResponse2(authenticated=True, claimed_id=req.claimed_device_id,
                            predicted_id=predicted_id, confidence=round(confidence, 4),
                            reason="Authentication successful")

client = TestClient(app_auth)

# 合法设备
test_iq = gen_iq(2, iq_len)
iq_flat = []
for i in range(iq_len):
    iq_flat.extend([float(test_iq[0, i]), float(test_iq[1, i])])

r1 = client.post("/auth/verify", json={"iq_data": iq_flat, "claimed_device_id": 2})
print(f"合法设备: {r1.json()}")

# 伪造设备 (声称是设备2, 实际是设备0)
test_iq0 = gen_iq(0, iq_len)
iq_flat0 = []
for i in range(iq_len):
    iq_flat0.extend([float(test_iq0[0, i]), float(test_iq0[1, i])])

r2 = client.post("/auth/verify", json={"iq_data": iq_flat0, "claimed_device_id": 2})
print(f"伪造设备: {r2.json()}")

---
## 5. 文件上传 (IQ数据)

实际场景中, IQ数据通常以文件形式上传。

In [ ]:
from fastapi import UploadFile, File
import io

app_upload = FastAPI(title="RFFI File Upload API")

rffi_model3 = SimpleRFFNet(n_classes=n_devices)
rffi_model3.load_state_dict(torch.load("rffi_model.pth", weights_only=True))
rffi_model3.eval()

@app_upload.post("/predict/file")
async def predict_from_file(file: UploadFile = File(...)):
    content = await file.read()
    iq = np.frombuffer(content, dtype=np.float32)
    iq = iq.reshape(2, -1)
    
    if iq.shape[1] < iq_len:
        return {"error": f"IQ length {iq.shape[1]} < required {iq_len}"}
    
    iq = iq[:, :iq_len]
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    
    with torch.no_grad():
        logits = rffi_model3(x)
        probs = torch.softmax(logits, dim=1)
        device_id = probs.argmax(1).item()
        confidence = probs.max().item()
    
    return {
        "filename": file.filename,
        "iq_length": iq.shape[1],
        "device_id": device_id,
        "confidence": round(confidence, 4),
    }

# 测试文件上传
client = TestClient(app_upload)
test_iq_bytes = gen_iq(3, iq_len).astype(np.float32).tobytes()

r = client.post("/predict/file", files={"file": ("iq_data.bin", test_iq_bytes, "application/octet-stream")})
print(f"文件上传预测: {r.json()}")

---
## 6. 中间件与CORS

### CORS (跨域资源共享)

前端页面调用API时需要配置CORS, 否则浏览器会阻止请求。

In [ ]:
from fastapi.middleware.cors import CORSMiddleware
from fastapi import Request
import time

app_full = FastAPI(title="RFFI Full API")

# CORS配置
app_full.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],       # 允许的源 (生产环境应限制)
    allow_credentials=True,
    allow_methods=["*"],       # 允许的HTTP方法
    allow_headers=["*"],       # 允许的请求头
)

# 自定义中间件: 请求计时
@app_full.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    process_time = time.time() - start
    response.headers["X-Process-Time"] = str(round(process_time * 1000, 2)) + "ms"
    return response

@app_full.get("/")
def root():
    return {"service": "RFFI API", "version": "1.0"}

client = TestClient(app_full)
r = client.get("/")
print(f"响应: {r.json()}")
print(f"处理时间: {r.headers.get('X-Process-Time')}")

In [ ]:
# 生命周期事件
print("=== 生命周期事件 ===")
print()
print("@app.on_event('startup')")
print("async def startup():")
print("    # 加载模型, 连接数据库等")
print("    model.load_state_dict(torch.load('model.pth'))")
print()
print("@app.on_event('shutdown')")
print("async def shutdown():")
print("    # 释放资源")
print("    pass")
print()
print("新版推荐使用 lifespan:")
print("from contextlib import asynccontextmanager")
print()
print("@asynccontextmanager")
print("async def lifespan(app):")
print("    # startup")
    print("    model = load_model()")
print("    yield")
print("    # shutdown")
print("    del model")

---
## 7. 完整RFFI服务示例

以下是一个可以直接运行的完整RFFI API服务。

In [ ]:
# 完整的 rffi_server.py
# 保存为独立文件后用 uvicorn rffi_server:app --reload 运行

server_code = '''
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional
import torch
import torch.nn as nn
import numpy as np
from contextlib import asynccontextmanager

# ---- 模型定义 ----
class SimpleRFFNet(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ---- 生命周期 ----
model = None
N_CLASSES = 5
IQ_LEN = 128

@asynccontextmanager
async def lifespan(app):
    global model
    model = SimpleRFFNet(n_classes=N_CLASSES)
    model.load_state_dict(torch.load("rffi_model.pth", weights_only=True))
    model.eval()
    print("Model loaded!")
    yield
    del model
    print("Model unloaded")

# ---- API ----
app = FastAPI(title="RFFI API", version="1.0", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictRequest(BaseModel):
    iq_data: List[float] = Field(..., min_length=256, max_length=256)

class PredictResponse(BaseModel):
    device_id: int
    confidence: float
    probabilities: List[float]

class AuthRequest(BaseModel):
    iq_data: List[float] = Field(..., min_length=256, max_length=256)
    claimed_device_id: int = Field(..., ge=0, lt=N_CLASSES)
    threshold: float = Field(0.7, ge=0, le=1)

@app.get("/")
def root():
    return {"service": "RFFI API", "version": "1.0", "n_devices": N_CLASSES}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    iq = np.array(req.iq_data, dtype=np.float32).reshape(2, IQ_LEN)
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
    return PredictResponse(
        device_id=probs.argmax(1).item(),
        confidence=round(probs.max().item(), 4),
        probabilities=[round(p, 4) for p in probs[0].numpy()]
    )

@app.post("/auth/verify")
def verify(req: AuthRequest):
    iq = np.array(req.iq_data, dtype=np.float32).reshape(2, IQ_LEN)
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)
        predicted = probs.argmax(1).item()
        confidence = probs.max().item()
    authenticated = (predicted == req.claimed_device_id and confidence >= req.threshold)
    return {
        "authenticated": authenticated,
        "predicted_id": predicted,
        "claimed_id": req.claimed_device_id,
        "confidence": round(confidence, 4),
    }

@app.post("/predict/file")
async def predict_file(file: UploadFile = File(...)):
    content = await file.read()
    iq = np.frombuffer(content, dtype=np.float32).reshape(2, -1)[:, :IQ_LEN]
    iq = iq / (np.std(iq) + 1e-8)
    x = torch.from_numpy(iq).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)
    return {"device_id": probs.argmax(1).item(), "confidence": round(probs.max().item(), 4)}
'''

print("完整服务代码已生成, 保存为 rffi_server.py 后运行:")
print("  uvicorn rffi_server:app --reload --host 0.0.0.0 --port 8000")
print()
print("API端点:")
print("  GET  /              -> 服务信息")
print("  POST /predict       -> IQ数据预测")
print("  POST /auth/verify   -> 设备认证")
print("  POST /predict/file  -> 文件上传预测")
print("  GET  /docs          -> Swagger文档")

In [ ]:
# 客户端调用示例
print("=== 客户端调用示例 ===")
print()
print("1. Python requests:")
print("   import requests")
print("   iq_data = [...]  # 256个float")
print("   r = requests.post('http://localhost:8000/predict', json={'iq_data': iq_data})")
print("   print(r.json())")
print()
print("2. curl:")
print("   curl -X POST http://localhost:8000/predict \\")
print("     -H 'Content-Type: application/json' \\")
print("     -d '{"iq_data": [0.1, 0.2, ...]}'")
print()
print("3. JavaScript fetch:")
print("   const response = await fetch('http://localhost:8000/predict', {")
print("     method: 'POST',")
print("     headers: {'Content-Type': 'application/json'},")
print("     body: JSON.stringify({iq_data: [...]})")
print("   });")
print("   const data = await response.json();")
print()
print("4. 文件上传:")
print("   with open('iq_data.bin', 'rb') as f:")
print("       r = requests.post('http://localhost:8000/predict/file', files={'file': f})")

In [ ]:
# 生产部署建议
print("=== 生产部署建议 ===")
print()
print("1. 运行方式:")
print("   开发: uvicorn app:app --reload")
print("   生产: gunicorn app:app -w 4 -k uvicorn.workers.UvicornWorker")
print()
print("2. Docker部署:")
print("   FROM python:3.11-slim")
print("   COPY . /app")
print("   RUN pip install fastapi uvicorn torch")
print("   CMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8000\"]")
print()
print("3. 性能优化:")
print("   - 模型推理用GPU: model.to('cuda')")
print("   - 批量推理: 积攒多个请求一起推理")
print("   - ONNX Runtime: 比PyTorch推理更快")
print("   - 模型量化: FP32 -> INT8, 速度提升2-4x")
print()
print("4. 安全:")
print("   - API Key认证")
print("   - HTTPS (用Nginx反向代理)")
print("   - 请求频率限制")
print("   - 输入大小限制")
print()
print("5. 监控:")
print("   - Prometheus + Grafana")
print("   - 日志: Python logging")
print("   - 健康检查: /health 端点")

In [ ]:
# FastAPI 速查表
print("=== FastAPI 速查表 ===")
print()
print("基础:")
print("  app = FastAPI()              # 创建应用")
print("  @app.get('/path')            # GET路由")
print("  @app.post('/path')           # POST路由")
print()
print("参数:")
print("  def func(item_id: int)       # 路径参数")
print("  def func(skip: int = 0)      # 查询参数")
print("  def func(data: BaseModel)    # 请求体")
print()
print("数据校验:")
print("  Field(..., min_length=1)     # 必填, 最小长度1")
print("  Field(None, ge=0, le=100)   # 可选, 0-100")
print("  Optional[str] = None        # 可选字段")
print()
print("响应:")
print("  response_model=Model         # 响应模型")
print("  HTTPException(404)           # 抛出异常")
print()
print("文件:")
print("  file: UploadFile = File(...) # 文件上传")
print()
print("运行:")
print("  uvicorn app:app --reload     # 开发")
print("  uvicorn app:app --host 0.0.0.0 --port 8000  # 生产")

---
## 总结

### FastAPI 核心概念

```
FastAPI
|-- 路由: @app.get / @app.post
|-- 参数: 路径参数 / 查询参数 / 请求体
|-- 校验: Pydantic BaseModel + Field
|-- 文档: 自动生成 /docs 和 /redoc
|-- 中间件: CORS / 自定义处理
|-- 生命周期: startup / shutdown
|-- 文件上传: UploadFile
|-- 异步: async/await
|-- 测试: TestClient
+-- 部署: Uvicorn / Gunicorn / Docker
```

### RFFI API 设计

| 端点 | 方法 | 功能 |
|------|------|------|
| `/predict` | POST | IQ数据预测设备ID |
| `/auth/verify` | POST | 设备认证 (带阈值) |
| `/predict/file` | POST | 文件上传预测 |
| `/model/info` | GET | 模型信息 |
| `/health` | GET | 健康检查 |

### 深度学习部署流程

```
训练模型 -> 保存权重 -> FastAPI加载 -> API端点推理 -> 客户端调用
```